In [1]:
import os

import numpy as np
import pandas as pd
import json
from datetime import datetime

/tmp/ipykernel_175297/914957945.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
def transform_time(df, col="time"):
    format = '%Y-%m-%d %H:%M:%S.%f'
    if col == 'time':
        df['time'] = df['time'].apply(lambda x: datetime.strptime(x, format))
    else:
        df[col] = df[col].apply(lambda x: datetime.strptime(x, format))
    return df

def second_difference(d2, d1):
    format = '%Y-%m-%d %H:%M:%S.%f'
    d2 = datetime.strptime(d2, format)
    d1 = datetime.strptime(d1, format)
    return (d2 - d1).seconds

def minutes_difference(d2, d1):
    return second_difference(d2, d1) / 60

In [3]:
# parse generic event

def parse_json_field(event_df, json_field):
    #transform data from str values to dict values
    event_df[json_field + "_parsed"] = event_df[json_field].apply(json.loads)
    #build dataframe from dict values
    event_json_df = pd.json_normalize(event_df[json_field + "_parsed"])
    #reset indexes to concat the dataframes along axis 1 (1:1)
    event_df.reset_index(drop=True, inplace=True)
    event_json_df.reset_index(drop=True, inplace=True)
    return event_json_df

def parse_generic_event(generic_interactions, event = None):
    if event:
        if isinstance(event, list):
            event_df = generic_interactions[generic_interactions['event'].isin(event)]
        else:
            event_df = generic_interactions[generic_interactions['event'] == event]
    else:
        event_df = generic_interactions.copy()
    event_df = event_df[['id', 'time', 'participant_id', 'data', 'event', 'event_uuid', 'participant_token']]
    event_df['time'] = event_df['time'].astype(str)
    event_json_df = parse_json_field(event_df, 'data')
    return pd.concat([event_df, event_json_df], axis=1)

def parse_keyboard_event(keyboard_interactions):
    event_df = keyboard_interactions.copy()
    event_df['time'] = event_df['time'].astype(str)
    event_json_df = parse_json_field(event_df, 'meta_data')
    return pd.concat([event_df, event_json_df], axis=1)

In [4]:
participants_groups = {
    "1": {"users": [1379, 1380], "user_ids": ['67ce982f285fac3f07bcf5e7', '67ce9cb4285fac3f07bcf5ec'], "documents": ["67ce983b285fac3f07bcf5e8"]},
    "2": {"users": [1376, 1377], "user_ids": ['67cb6627285fac3f07bcf59b', '67cb6623285fac3f07bcf59a'], "documents": ["67c97527285fac3f07bcf587"]},
    "3": {"users": [1378, 1374], "user_ids": ['67cc9f69285fac3f07bcf5c8', '67cdfa30285fac3f07bcf5cc'], "documents": ["67c88f88285fac3f07bcf585"]},
    "4": {"users": [1383, 1384, 1385], "user_ids": ['67bc761a7a670b68746905a0', '67d444dd0b9539e69a3a86bb', '67bc43607a670b6874690577'], "documents": ["67d43b390b9539e69a3a86b9"]},
    "5": {"users": [1387, 1390], "user_ids": ['67d5d8120b9539e69a3a86d7', '67e18e95edfe680c1b0f169f'], "documents": ["67d5d8160b9539e69a3a86d8"]},
    "6": {"users": [1388, 1389], "user_ids": ['67d82dc4edfe680c1b0f1642', '67d82e0aedfe680c1b0f1643'], "documents": ["67d08e87285fac3f07bcf64b"]},
    "7": {"users": [2456, 2460], "user_ids": ['68835b0587cf5fec60dc8020', '6887b97087cf5fec60dc8060'], "documents": ["68833fd687cf5fec60dc801d"]},
    "8": {"users": [2453, 2455], "user_ids": ['687de5d187cf5fec60dc7fef', '6880e0f187cf5fec60dc7ffe'], "documents": ["6879e8fd87cf5fec60dc7fe1"]},
    "9": {"users": [2461, 2464, 2457], "user_ids": ['6888eec887cf5fec60dc8085', '688a52ad87cf5fec60dc8096', '688b1c7887cf5fec60dc80b0'], "documents": ["688364bf87cf5fec60dc8026"]},
    "10": {"users": [2459, 2465], "user_ids": ['6888bbce87cf5fec60dc8072', '688bdb6387cf5fec60dc80d0'], "documents": ["6877c26e87cf5fec60dc7fda"]},
    "11": {"users": [2463, 2475], "user_ids": ['6889e71f87cf5fec60dc8094', '68a75365ecc07d2944852275'], "documents": ["6889e57087cf5fec60dc8092"]},
    "12": {"users": [2468, 2469], "user_ids": ['6898407be90ffc190a96c287', '689c3c25e90ffc190a96c28b'], "documents": ["688c6add87cf5fec60dc80ef"]},
    "13": {"users": [2467, 2476], "user_ids": ['68a5a0bde90ffc190a96c2ff', '68a59766e90ffc190a96c2f4'], "documents": ["68907a6e87cf5fec60dc8107"]},
    "14": {"users": [2470, 2471], "user_ids": ['689e2181e90ffc190a96c2df', '689e22d5e90ffc190a96c2e0'], "documents": ["6896071ee90ffc190a96c285"]},
} 


participants_id = [1378, 1374, 1376, 1377, 1379, 1384, 1383, 1385, 1387, 1388, 1389, 1390, 1380]
participants_id_wo_directusers = [1378, 1374, 1376, 1377, 1379, 1387, 1388, 1389, 1390, 1380] #removed 1383 1384 1385
participants_id_study_extended = [2456, 2460, 2453, 2455, 2461, 2464, 2457, 2459, 2465, 2463, 2475, 2468, 2469, 2467, 2476, 2470, 2471]

# join lists
participants_id = participants_id + participants_id_study_extended
participants_id_wo_directusers = participants_id_wo_directusers + participants_id_study_extended

In [5]:
finished_participants_file = '03_finished_participants.csv'
finished_participants_generic = pd.read_csv(os.path.join('output', 'intermediate', finished_participants_file), sep='\t')
display(finished_participants_generic.head())

,Unnamed: 0.1,Unnamed: 0,id,event_uuid,name,time,event,data,participant_id,participant_token,participant_state,data_parsed,document_id
0,0,164,276174.0,cb251ecc-b22b-4ef9-af15-415f3ab90c25,Prototype,2025-03-07 21:32:17.982000,USER_AGENT,"""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_...",1376,a5bbafd1-4ef0-49e4-95bd-65170953633c,finished,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN
1,1,166,276176.0,17b21821-b077-4b06-9db8-b341d36d0301,Prototype,2025-03-07 21:33:27.678000,USER_LOGIN,"{""username"": ""wolfova"", ""participant_token"": ""...",1376,a5bbafd1-4ef0-49e4-95bd-65170953633c,finished,"{'username': 'wolfova', 'participant_token': '...",NaN
2,2,171,276181.0,c55551a8-7950-417a-aa64-9c323aaee006,Prototype,2025-03-07 21:33:49.699000,DOCUMENT_JOIN,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern...",1376,a5bbafd1-4ef0-49e4-95bd-65170953633c,finished,"{'user_id': '67cb6627285fac3f07bcf59b', 'usern...",67c97527285fac3f07bcf587
3,3,172,276182.0,65ca58d2-568f-4b16-af2d-718c2e633485,Prototype,2025-03-07 21:33:49.808000,DOCUMENT_VIEW,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern...",1376,a5bbafd1-4ef0-49e4-95bd-65170953633c,finished,"{'user_id': '67cb6627285fac3f07bcf59b', 'usern...",67c97527285fac3f07bcf587
4,4,173,276183.0,20b1b819-a53b-4619-87e9-fe11dcba2855,Prototype,2025-03-07 21:33:59.815000,DOCUMENT_TEXT_UPDATE,"{""text"": ""<p>Write your text here</p>"", ""user_...",1376,a5bbafd1-4ef0-49e4-95bd-65170953633c,finished,"{'text': '<p>Write your text here</p>', 'user_...",67c97527285fac3f07bcf587


In [6]:
finished_participants_file = '03_finished_participants_keyboard.csv'
finished_participants_keyboard = pd.read_csv(os.path.join('output', 'intermediate', finished_participants_file), sep='\t')
display(finished_participants_keyboard.head())

,Unnamed: 0.1,Unnamed: 0,id,condition_id,name,event_uuid,time,event,alt_key,code,...,location,meta_key,repeat,shift_key,meta_data,participant_id,participant_token,participant_state,meta_data_parsed,document_id
0,0,19,20.0,216.0,Prototype,23e508b6-6882-4d22-8b23-6517f6c8529f,2025-03-07 21:34:10.640000,keydown,False,Backspace,...,0.0,False,False,False,"{""source"": ""editor"", ""user_id"": ""67cb6627285fa...",1376,a5bbafd1-4ef0-49e4-95bd-65170953633c,finished,"{'source': 'editor', 'user_id': '67cb6627285fa...",67c97527285fac3f07bcf587
1,1,20,21.0,216.0,Prototype,27b03e7a-282a-4963-87d1-5d947ab9443c,2025-03-07 21:36:57.981000,keydown,False,Backspace,...,0.0,False,False,False,"{""source"": ""editor"", ""user_id"": ""67cb6627285fa...",1376,a5bbafd1-4ef0-49e4-95bd-65170953633c,finished,"{'source': 'editor', 'user_id': '67cb6627285fa...",67c97527285fac3f07bcf587
2,2,21,22.0,216.0,Prototype,80a90e34-7c35-413a-bf9a-13296a750b39,2025-03-07 21:38:20.585000,keydown,False,ShiftLeft,...,1.0,False,False,True,"{""source"": ""editor"", ""user_id"": ""67cb6627285fa...",1376,a5bbafd1-4ef0-49e4-95bd-65170953633c,finished,"{'source': 'editor', 'user_id': '67cb6627285fa...",67c97527285fac3f07bcf587
3,3,22,23.0,216.0,Prototype,4195cdfa-67e8-4a80-a430-c051cc1f6bd8,2025-03-07 21:38:21.048000,keydown,False,KeyT,...,0.0,False,False,True,"{""source"": ""editor"", ""user_id"": ""67cb6627285fa...",1376,a5bbafd1-4ef0-49e4-95bd-65170953633c,finished,"{'source': 'editor', 'user_id': '67cb6627285fa...",67c97527285fac3f07bcf587
4,4,23,24.0,216.0,Prototype,62115764-298f-4649-a7c0-97683ac2dc61,2025-03-07 21:38:21.794000,keydown,False,KeyH,...,0.0,False,False,False,"{""source"": ""editor"", ""user_id"": ""67cb6627285fa...",1376,a5bbafd1-4ef0-49e4-95bd-65170953633c,finished,"{'source': 'editor', 'user_id': '67cb6627285fa...",67c97527285fac3f07bcf587


In [7]:
finished_participants_generic_reduced = finished_participants_generic[["id", "event_uuid", "time", "event", "participant_id", "document_id", "data"]]
finished_participants_keyboard_reduced = finished_participants_keyboard[["id", "event_uuid", "time", "event", "participant_id", "document_id"]]

finished_participants_combined = pd.concat([finished_participants_generic_reduced, finished_participants_keyboard_reduced])

display(finished_participants_combined)
transform_time(finished_participants_combined)

finished_participants_combined = finished_participants_combined.sort_values(by='time', ascending=True)
display(finished_participants_combined)

finished_participants_combined.to_csv(os.path.join('output', 'intermediate', '03_finished_participants_combined.tsv'), sep="\t")

,id,event_uuid,time,event,participant_id,document_id,data
0,276174.0,cb251ecc-b22b-4ef9-af15-415f3ab90c25,2025-03-07 21:32:17.982000,USER_AGENT,1376,NaN,"""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_..."
1,276176.0,17b21821-b077-4b06-9db8-b341d36d0301,2025-03-07 21:33:27.678000,USER_LOGIN,1376,NaN,"{""username"": ""wolfova"", ""participant_token"": ""..."
2,276181.0,c55551a8-7950-417a-aa64-9c323aaee006,2025-03-07 21:33:49.699000,DOCUMENT_JOIN,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern..."
3,276182.0,65ca58d2-568f-4b16-af2d-718c2e633485,2025-03-07 21:33:49.808000,DOCUMENT_VIEW,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern..."
4,276183.0,20b1b819-a53b-4619-87e9-fe11dcba2855,2025-03-07 21:33:59.815000,DOCUMENT_TEXT_UPDATE,1376,67c97527285fac3f07bcf587,"{""text"": ""<p>Write your text here</p>"", ""user_..."
...,...,...,...,...,...,...,...
33549,53681.0,bee629df-5038-4bf4-960a-a32d026c692b,2025-08-22 18:20:06.878000,keydown,2475,6889e57087cf5fec60dc8092,NaN
33550,53682.0,0d756113-f5f1-4e35-8a79-151ab4f3a7a5,2025-08-22 18:21:00.031000,keydown,2475,6889e57087cf5fec60dc8092,NaN
33551,53683.0,9d3fc61c-846c-49be-929d-f1a095226568,2025-08-22 18:21:00.646000,keydown,2475,6889e57087cf5fec60dc8092,NaN
33552,53684.0,1c7b5920-44a5-477b-be8d-71dd030e878c,2025-08-22 18:21:11.375000,keydown,2475,6889e57087cf5fec60dc8092,NaN


,id,event_uuid,time,event,participant_id,document_id,data
0,276174.0,cb251ecc-b22b-4ef9-af15-415f3ab90c25,2025-03-07 21:32:17.982,USER_AGENT,1376,NaN,"""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_..."
1,276176.0,17b21821-b077-4b06-9db8-b341d36d0301,2025-03-07 21:33:27.678,USER_LOGIN,1376,NaN,"{""username"": ""wolfova"", ""participant_token"": ""..."
2,276181.0,c55551a8-7950-417a-aa64-9c323aaee006,2025-03-07 21:33:49.699,DOCUMENT_JOIN,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern..."
3,276182.0,65ca58d2-568f-4b16-af2d-718c2e633485,2025-03-07 21:33:49.808,DOCUMENT_VIEW,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern..."
4,276183.0,20b1b819-a53b-4619-87e9-fe11dcba2855,2025-03-07 21:33:59.815,DOCUMENT_TEXT_UPDATE,1376,67c97527285fac3f07bcf587,"{""text"": ""<p>Write your text here</p>"", ""user_..."
...,...,...,...,...,...,...,...
6639,284568.0,98ba293b-978c-4cdd-9bd6-b814f1fab9df,2025-08-22 18:29:05.963,DOCUMENT_VIEW,2475,6889e57087cf5fec60dc8092,"{""user_id"": ""68a75365ecc07d2944852275"", ""usern..."
6640,284569.0,1f8d5ecd-c7ca-48e5-86e7-86c9d3d639da,2025-08-22 18:29:10.543,FINAL_PROCEED_TEXT,2475,NaN,"{""text"": ""<h1>Hauptseminar WiSe26</h1><p><br><..."
6641,284570.0,c292fbcb-986f-4d19-8ffc-7f2e47c583a3,2025-08-22 18:29:15.735,DOCUMENT_SAVE,2475,6889e57087cf5fec60dc8092,"{""user_id"": ""68a75365ecc07d2944852275"", ""usern..."
6642,284571.0,3385c09e-5f47-4a71-bbd6-8fdf000bbfa2,2025-08-22 18:29:16.353,DOCUMENT_TEXT_UPDATE,2475,6889e57087cf5fec60dc8092,"{""text"": ""<h1>Hauptseminar WiSe26</h1><p><br><..."


In [8]:
#finished_participants_combined = parse_json_field(finished_participants_combined[finished_participants_combined["data"].notna()], "data")

# keep all events that were related to a document
finished_participants_combined = finished_participants_combined[finished_participants_combined["document_id"].notna()]
display(finished_participants_combined.head(20))

ai_events = ["REPLY_AI", "SUGGESTION_INSERT_AFTER", "SUGGESTION_TAKE_OVER", "SUGGESTION_COPY_TO_CLIPBOARD"]

finished_participants_combined["ai_id"] = np.nan

# defining function to check price
def add_ai_id(row):
    if row["event"] in ai_events:
        if row["data"] and row["data"] is not None:
            data = json.loads(row["data"])        
            if row["event"] != "REPLY_AI":
                if data["card"]["data"]["user"]["type"] == "ai":
                    row["ai_id"] = data["card"]["data"]["user"]["id"]
            else:
                if data["reply"]["data"]["user"]["type"] == "ai":
                    row["ai_id"] = data["reply"]["data"]["user"]["id"]
    return row
        
# apply functions to rows to add ai_id
finished_participants_combined = finished_participants_combined.apply(add_ai_id, axis=1)
display(finished_participants_combined)


,id,event_uuid,time,event,participant_id,document_id,data
2,276181.0,c55551a8-7950-417a-aa64-9c323aaee006,2025-03-07 21:33:49.699,DOCUMENT_JOIN,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern..."
3,276182.0,65ca58d2-568f-4b16-af2d-718c2e633485,2025-03-07 21:33:49.808,DOCUMENT_VIEW,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern..."
4,276183.0,20b1b819-a53b-4619-87e9-fe11dcba2855,2025-03-07 21:33:59.815,DOCUMENT_TEXT_UPDATE,1376,67c97527285fac3f07bcf587,"{""text"": ""<p>Write your text here</p>"", ""user_..."
0,20.0,23e508b6-6882-4d22-8b23-6517f6c8529f,2025-03-07 21:34:10.640,keydown,1376,67c97527285fac3f07bcf587,NaN
5,276185.0,c11a2375-4af0-44d1-9d3a-3eb6d14f6d5a,2025-03-07 21:34:20.634,DOCUMENT_TEXT_UPDATE,1376,67c97527285fac3f07bcf587,"{""text"": ""<p><br></p>"", ""user_id"": ""67cb662728..."
6,276191.0,476378c2-4cbf-4a06-933c-4750219a3eec,2025-03-07 21:35:54.976,TASKS_VIEW,1376,67c97527285fac3f07bcf587,"{""source"": ""topbar"", ""user_id"": ""67cb6627285fa..."
7,276192.0,dfaeca38-1398-420c-b4f0-3666f1546b21,2025-03-07 21:35:57.717,TASK_CREATE,1376,67c97527285fac3f07bcf587,"{""source"": ""sidebar"", ""user_id"": ""67cb6627285f..."
8,276195.0,30b8d689-da4f-41e9-a050-e78f94fe6d29,2025-03-07 21:36:02.732,TASK_CREATE_CANCEL,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern..."
9,276196.0,af7afb6d-8b10-4b26-8b6b-5f98c7d04e45,2025-03-07 21:36:04.282,TASKS_VIEW_CLOSE,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern..."
10,276197.0,ca48e73d-2755-4e25-a612-4d14b88fb192,2025-03-07 21:36:06.132,TASKS_VIEW,1376,67c97527285fac3f07bcf587,"{""source"": ""topbar"", ""user_id"": ""67cb6627285fa..."


,id,event_uuid,time,event,participant_id,document_id,data,ai_id
2,276181.0,c55551a8-7950-417a-aa64-9c323aaee006,2025-03-07 21:33:49.699,DOCUMENT_JOIN,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern...",NaN
3,276182.0,65ca58d2-568f-4b16-af2d-718c2e633485,2025-03-07 21:33:49.808,DOCUMENT_VIEW,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern...",NaN
4,276183.0,20b1b819-a53b-4619-87e9-fe11dcba2855,2025-03-07 21:33:59.815,DOCUMENT_TEXT_UPDATE,1376,67c97527285fac3f07bcf587,"{""text"": ""<p>Write your text here</p>"", ""user_...",NaN
0,20.0,23e508b6-6882-4d22-8b23-6517f6c8529f,2025-03-07 21:34:10.640,keydown,1376,67c97527285fac3f07bcf587,NaN,NaN
5,276185.0,c11a2375-4af0-44d1-9d3a-3eb6d14f6d5a,2025-03-07 21:34:20.634,DOCUMENT_TEXT_UPDATE,1376,67c97527285fac3f07bcf587,"{""text"": ""<p><br></p>"", ""user_id"": ""67cb662728...",NaN
...,...,...,...,...,...,...,...,...
6637,284566.0,bbbe36d6-36c3-4686-911a-0e6f43541de2,2025-08-22 18:28:58.108,DOCUMENT_SELECT,2475,6889e57087cf5fec60dc8092,"{""user_id"": ""68a75365ecc07d2944852275"", ""usern...",NaN
6638,284567.0,e63d110b-0ab4-4a14-8866-b5aeb8b44ccb,2025-08-22 18:28:58.994,DOCUMENT_OPEN,2475,6889e57087cf5fec60dc8092,"{""user_id"": ""68a75365ecc07d2944852275"", ""usern...",NaN
6639,284568.0,98ba293b-978c-4cdd-9bd6-b814f1fab9df,2025-08-22 18:29:05.963,DOCUMENT_VIEW,2475,6889e57087cf5fec60dc8092,"{""user_id"": ""68a75365ecc07d2944852275"", ""usern...",NaN
6641,284570.0,c292fbcb-986f-4d19-8ffc-7f2e47c583a3,2025-08-22 18:29:15.735,DOCUMENT_SAVE,2475,6889e57087cf5fec60dc8092,"{""user_id"": ""68a75365ecc07d2944852275"", ""usern...",NaN


In [9]:
display(finished_participants_combined[finished_participants_combined["ai_id"].notna()].head(20))
display(finished_participants_combined.head(20))

# compute incremental ids for AIs (grouping)
finished_participants_combined["ai"] = finished_participants_combined.groupby("ai_id").ngroup()

display(finished_participants_combined[finished_participants_combined["ai_id"].notna()])
display(finished_participants_combined.head(20))

switches_without_ai = []
switches_with_ai = []

for idx, group_id in enumerate(participants_groups):
    #skip group 4 since they used the app w/o logging
    if int(group_id) == 4: 
        continue
            
    document_id = participants_groups[group_id]["documents"][0]
    
    group_events = finished_participants_combined[finished_participants_combined["document_id"] == document_id]
    
    #display("SWITCHES WITHOUT AI")
    group_events["switch"] = group_events["participant_id"].diff()
    switches = group_events[group_events['switch'] != 0]
    #display(group_events[group_events['switch'] != 0])
    
    switches_without_ai.append({"group_id": group_id, "switches": len(switches)})
    
    #display("SWITCHES WITH AI")
    group_events_ai_switches = finished_participants_combined[finished_participants_combined["document_id"] == document_id]
    
    # set ai id as participant id    
    group_events_ai_switches['participant_id'][(group_events_ai_switches['ai'].notna())] = group_events_ai_switches['ai']
    
    group_events_ai_switches["switch"] = group_events_ai_switches["participant_id"].diff()

    switches_ai = group_events_ai_switches[group_events_ai_switches['switch'] != 0]

    #display(group_events_ai_switches[group_events_ai_switches['switch'] != 0])
    
    switches_with_ai.append({"group_id": group_id, "switches": len(switches_ai)})

    
display(switches_with_ai)
display(switches_without_ai)


,id,event_uuid,time,event,participant_id,document_id,data,ai_id
52,276239.0,31658bce-0fbf-4559-9541-0242b1f07006,2025-03-07 21:42:34.724,SUGGESTION_INSERT_AFTER,1377,67c97527285fac3f07bcf587,"{""card"": {""id"": ""h9dJoEjrePKz1x9XOxrxx"", ""data...",ai-author
98,276285.0,b0449168-c323-4422-87a9-b21ded2e7520,2025-03-07 21:44:21.621,SUGGESTION_INSERT_AFTER,1376,67c97527285fac3f07bcf587,"{""card"": {""id"": ""5Plaf8nQ31ghtWn4OAFbg"", ""data...",ai-author
160,276347.0,ce9eb039-b494-481b-8dbf-5329df65e520,2025-03-07 21:46:31.484,SUGGESTION_INSERT_AFTER,1376,67c97527285fac3f07bcf587,"{""card"": {""id"": ""T1140AO9etlFPy5sX0me-"", ""data...",ai-author
200,276387.0,07f107d3-e7ba-4491-9556-25bd48034de4,2025-03-07 21:49:31.354,SUGGESTION_INSERT_AFTER,1376,67c97527285fac3f07bcf587,"{""card"": {""id"": ""EiPiEDL-QXWG5qiQOj9tm"", ""data...",ai-author
359,276548.0,7f8d99ff-7817-47c7-ac7c-38f8035b8fd2,2025-03-09 21:16:15.493,REPLY_AI,1374,67c88f88285fac3f07bcf585,"{""card"": {}, ""reply"": {""id"": ""mKXUvYDt64Vu7lTG...",67cdfef4285fac3f07bcf5d3
372,276561.0,053179a4-8667-40fe-92c5-865f39733206,2025-03-09 21:18:22.747,REPLY_AI,1374,67c88f88285fac3f07bcf585,"{""card"": {""id"": ""XzJllbkbnRCR-EZRbYc37"", ""data...",67cdfef4285fac3f07bcf5d3
384,276573.0,94343183-fe1e-4a2f-9720-bf57e7884a92,2025-03-09 21:19:50.329,REPLY_AI,1374,67c88f88285fac3f07bcf585,"{""card"": {""id"": ""XzJllbkbnRCR-EZRbYc37"", ""data...",67cdfef4285fac3f07bcf5d3
393,276582.0,58a53257-79db-44c0-8a72-049dd12998af,2025-03-09 21:20:49.162,REPLY_AI,1374,67c88f88285fac3f07bcf585,"{""card"": {""id"": ""XzJllbkbnRCR-EZRbYc37"", ""data...",67cdfef4285fac3f07bcf5d3
453,276642.0,2634704f-a9d6-4adf-bacd-90301dd61512,2025-03-09 21:29:45.705,REPLY_AI,1374,67c88f88285fac3f07bcf585,"{""card"": {}, ""reply"": {""id"": ""HpJz8EbDof1-5CtI...",67cdfef4285fac3f07bcf5d3
576,276765.0,3b2512d3-0bc0-4405-babd-04d06d19af2c,2025-03-10 08:05:01.122,REPLY_AI,1379,67ce983b285fac3f07bcf5e8,"{""card"": {""id"": ""XQt5XrkOapXdkdaV-J3Cn"", ""data...",ai-author


,id,event_uuid,time,event,participant_id,document_id,data,ai_id
2,276181.0,c55551a8-7950-417a-aa64-9c323aaee006,2025-03-07 21:33:49.699,DOCUMENT_JOIN,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern...",NaN
3,276182.0,65ca58d2-568f-4b16-af2d-718c2e633485,2025-03-07 21:33:49.808,DOCUMENT_VIEW,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern...",NaN
4,276183.0,20b1b819-a53b-4619-87e9-fe11dcba2855,2025-03-07 21:33:59.815,DOCUMENT_TEXT_UPDATE,1376,67c97527285fac3f07bcf587,"{""text"": ""<p>Write your text here</p>"", ""user_...",NaN
0,20.0,23e508b6-6882-4d22-8b23-6517f6c8529f,2025-03-07 21:34:10.640,keydown,1376,67c97527285fac3f07bcf587,NaN,NaN
5,276185.0,c11a2375-4af0-44d1-9d3a-3eb6d14f6d5a,2025-03-07 21:34:20.634,DOCUMENT_TEXT_UPDATE,1376,67c97527285fac3f07bcf587,"{""text"": ""<p><br></p>"", ""user_id"": ""67cb662728...",NaN
6,276191.0,476378c2-4cbf-4a06-933c-4750219a3eec,2025-03-07 21:35:54.976,TASKS_VIEW,1376,67c97527285fac3f07bcf587,"{""source"": ""topbar"", ""user_id"": ""67cb6627285fa...",NaN
7,276192.0,dfaeca38-1398-420c-b4f0-3666f1546b21,2025-03-07 21:35:57.717,TASK_CREATE,1376,67c97527285fac3f07bcf587,"{""source"": ""sidebar"", ""user_id"": ""67cb6627285f...",NaN
8,276195.0,30b8d689-da4f-41e9-a050-e78f94fe6d29,2025-03-07 21:36:02.732,TASK_CREATE_CANCEL,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern...",NaN
9,276196.0,af7afb6d-8b10-4b26-8b6b-5f98c7d04e45,2025-03-07 21:36:04.282,TASKS_VIEW_CLOSE,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern...",NaN
10,276197.0,ca48e73d-2755-4e25-a612-4d14b88fb192,2025-03-07 21:36:06.132,TASKS_VIEW,1376,67c97527285fac3f07bcf587,"{""source"": ""topbar"", ""user_id"": ""67cb6627285fa...",NaN


,id,event_uuid,time,event,participant_id,document_id,data,ai_id,ai
52,276239.0,31658bce-0fbf-4559-9541-0242b1f07006,2025-03-07 21:42:34.724,SUGGESTION_INSERT_AFTER,1377,67c97527285fac3f07bcf587,"{""card"": {""id"": ""h9dJoEjrePKz1x9XOxrxx"", ""data...",ai-author,24.0
98,276285.0,b0449168-c323-4422-87a9-b21ded2e7520,2025-03-07 21:44:21.621,SUGGESTION_INSERT_AFTER,1376,67c97527285fac3f07bcf587,"{""card"": {""id"": ""5Plaf8nQ31ghtWn4OAFbg"", ""data...",ai-author,24.0
160,276347.0,ce9eb039-b494-481b-8dbf-5329df65e520,2025-03-07 21:46:31.484,SUGGESTION_INSERT_AFTER,1376,67c97527285fac3f07bcf587,"{""card"": {""id"": ""T1140AO9etlFPy5sX0me-"", ""data...",ai-author,24.0
200,276387.0,07f107d3-e7ba-4491-9556-25bd48034de4,2025-03-07 21:49:31.354,SUGGESTION_INSERT_AFTER,1376,67c97527285fac3f07bcf587,"{""card"": {""id"": ""EiPiEDL-QXWG5qiQOj9tm"", ""data...",ai-author,24.0
359,276548.0,7f8d99ff-7817-47c7-ac7c-38f8035b8fd2,2025-03-09 21:16:15.493,REPLY_AI,1374,67c88f88285fac3f07bcf585,"{""card"": {}, ""reply"": {""id"": ""mKXUvYDt64Vu7lTG...",67cdfef4285fac3f07bcf5d3,1.0
...,...,...,...,...,...,...,...,...,...
6473,284402.0,6f15c591-b7ac-40cb-b47b-3d5a77772f9c,2025-08-22 17:14:32.718,SUGGESTION_INSERT_AFTER,2475,6889e57087cf5fec60dc8092,"{""card"": {""id"": ""oucHjS_N0NE6Ec1wd4bI-"", ""data...",68931d9887cf5fec60dc810f,19.0
6479,284408.0,11a68622-7bc8-47bc-ab65-a455bdcf37fa,2025-08-22 17:15:47.279,SUGGESTION_COPY_TO_CLIPBOARD,2475,6889e57087cf5fec60dc8092,"{""card"": {""id"": ""f8-R6jTXSrds6QlCxyNKM"", ""data...",68931d9887cf5fec60dc810f,19.0
6491,284420.0,a08ff692-f5d2-4c84-b04e-177c6901bd0b,2025-08-22 17:18:02.061,REPLY_AI,2475,6889e57087cf5fec60dc8092,"{""card"": {}, ""reply"": {""id"": ""djNYydo1Jx4uVPsi...",68931d9887cf5fec60dc810f,19.0
6539,284468.0,2d544194-aa71-4fe0-8737-13e246c4bba3,2025-08-22 17:34:07.498,SUGGESTION_COPY_TO_CLIPBOARD,2475,6889e57087cf5fec60dc8092,"{""card"": {""id"": ""T6dovEXM5kHzPLUczuYjE"", ""data...",688b2aa687cf5fec60dc80b3,16.0


,id,event_uuid,time,event,participant_id,document_id,data,ai_id,ai
2,276181.0,c55551a8-7950-417a-aa64-9c323aaee006,2025-03-07 21:33:49.699,DOCUMENT_JOIN,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern...",NaN,NaN
3,276182.0,65ca58d2-568f-4b16-af2d-718c2e633485,2025-03-07 21:33:49.808,DOCUMENT_VIEW,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern...",NaN,NaN
4,276183.0,20b1b819-a53b-4619-87e9-fe11dcba2855,2025-03-07 21:33:59.815,DOCUMENT_TEXT_UPDATE,1376,67c97527285fac3f07bcf587,"{""text"": ""<p>Write your text here</p>"", ""user_...",NaN,NaN
0,20.0,23e508b6-6882-4d22-8b23-6517f6c8529f,2025-03-07 21:34:10.640,keydown,1376,67c97527285fac3f07bcf587,NaN,NaN,NaN
5,276185.0,c11a2375-4af0-44d1-9d3a-3eb6d14f6d5a,2025-03-07 21:34:20.634,DOCUMENT_TEXT_UPDATE,1376,67c97527285fac3f07bcf587,"{""text"": ""<p><br></p>"", ""user_id"": ""67cb662728...",NaN,NaN
6,276191.0,476378c2-4cbf-4a06-933c-4750219a3eec,2025-03-07 21:35:54.976,TASKS_VIEW,1376,67c97527285fac3f07bcf587,"{""source"": ""topbar"", ""user_id"": ""67cb6627285fa...",NaN,NaN
7,276192.0,dfaeca38-1398-420c-b4f0-3666f1546b21,2025-03-07 21:35:57.717,TASK_CREATE,1376,67c97527285fac3f07bcf587,"{""source"": ""sidebar"", ""user_id"": ""67cb6627285f...",NaN,NaN
8,276195.0,30b8d689-da4f-41e9-a050-e78f94fe6d29,2025-03-07 21:36:02.732,TASK_CREATE_CANCEL,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern...",NaN,NaN
9,276196.0,af7afb6d-8b10-4b26-8b6b-5f98c7d04e45,2025-03-07 21:36:04.282,TASKS_VIEW_CLOSE,1376,67c97527285fac3f07bcf587,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern...",NaN,NaN
10,276197.0,ca48e73d-2755-4e25-a612-4d14b88fb192,2025-03-07 21:36:06.132,TASKS_VIEW,1376,67c97527285fac3f07bcf587,"{""source"": ""topbar"", ""user_id"": ""67cb6627285fa...",NaN,NaN


/tmp/ipykernel_175297/3310570086.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  group_events["switch"] = group_events["participant_id"].diff()
/tmp/ipykernel_175297/3310570086.py:33: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead

[{'group_id': '1', 'switches': 580},
 {'group_id': '2', 'switches': 99},
 {'group_id': '3', 'switches': 35},
 {'group_id': '5', 'switches': 42},
 {'group_id': '6', 'switches': 221},
 {'group_id': '7', 'switches': 136},
 {'group_id': '8', 'switches': 10},
 {'group_id': '9', 'switches': 337},
 {'group_id': '10', 'switches': 20},
 {'group_id': '11', 'switches': 18},
 {'group_id': '12', 'switches': 69},
 {'group_id': '13', 'switches': 0},
 {'group_id': '14', 'switches': 993}]

[{'group_id': '1', 'switches': 559},
 {'group_id': '2', 'switches': 94},
 {'group_id': '3', 'switches': 5},
 {'group_id': '5', 'switches': 32},
 {'group_id': '6', 'switches': 172},
 {'group_id': '7', 'switches': 62},
 {'group_id': '8', 'switches': 4},
 {'group_id': '9', 'switches': 280},
 {'group_id': '10', 'switches': 4},
 {'group_id': '11', 'switches': 2},
 {'group_id': '12', 'switches': 61},
 {'group_id': '13', 'switches': 0},
 {'group_id': '14', 'switches': 987}]

In [10]:
switches_df = pd.DataFrame(data=switches_with_ai, columns=["group_id", "switches"])
display(switches_df)

display(switches_df["switches"].describe())

,group_id,switches
0,1,580
1,2,99
2,3,35
3,5,42
4,6,221
5,7,136
6,8,10
7,9,337
8,10,20
9,11,18


count     13.000000
mean     196.923077
std      291.205386
min        0.000000
25%       20.000000
50%       69.000000
75%      221.000000
max      993.000000
Name: switches, dtype: float64

In [11]:
switches_df = pd.DataFrame(data=switches_without_ai, columns=["group_id", "switches"])
display(switches_df)

display(switches_df["switches"].describe())

,group_id,switches
0,1,559
1,2,94
2,3,5
3,5,32
4,6,172
5,7,62
6,8,4
7,9,280
8,10,4
9,11,2


count     13.000000
mean     174.000000
std      291.337719
min        0.000000
25%        4.000000
50%       61.000000
75%      172.000000
max      987.000000
Name: switches, dtype: float64